# [12-6강] MLP vs CNN 성능 비교 - 실습

In [1]:
import torch
torch.set_num_threads(1)
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import random

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)

torch.set_printoptions(precision=4, sci_mode=False)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


device: cpu


## 문제 1. MLP와 CNN 모델 준비하기

같은 입력 이미지에 대해 MLP와 CNN이 모두 2개 class logits를 출력하도록 모델을 준비합니다.

In [2]:
x = torch.randn(4, 1, 8, 8)
# TODO: 두 모델을 완성하세요.
mlp = nn.Sequential(
    nn.Flatten(), nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 2)
)
cnn = nn.Sequential(
    nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(),
    nn.Flatten(), nn.Linear(4 * 8 * 8, 2)
)

for name, model in [('MLP', mlp), ('CNN', cnn)]:
    try:
        print(name, model(x).shape)
    except RuntimeError as e:
        print(name, '수정 필요:', str(e).split('\\n')[0])


MLP torch.Size([4, 2])
CNN torch.Size([4, 2])


## 문제 2. 두 모델을 같은 함수로 학습하기

공통 train/evaluate 함수를 사용해 MLP와 CNN을 같은 기준으로 학습합니다.

In [6]:
def make_toy_images(n=48, size=8):
    # class 0: 세로선, class 1: 가로선
    x = torch.zeros(n, 1, size, size)
    y = torch.zeros(n, dtype=torch.long)
    for i in range(n):
        if i % 2 == 0:
            x[i, 0, :, 3:5] = 1.0
            y[i] = 0
        else:
            x[i, 0, 3:5, :] = 1.0
            y[i] = 1
    x += 0.05 * torch.randn_like(x)
    return x, y

images, labels = make_toy_images()
train_ds = TensorDataset(images[:40], labels[:40])
valid_ds = TensorDataset(images[40:], labels[40:])
valid_loader = DataLoader(valid_ds, batch_size=8, shuffle=False)

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def make_train_loader(seed):
    generator = torch.Generator().manual_seed(seed)
    return DataLoader(
        train_ds, batch_size=8, shuffle=True, generator=generator
    )

model_builders = {
    'MLP': lambda: nn.Sequential(
        nn.Flatten(), nn.Linear(64, 16), nn.ReLU(), nn.Linear(16, 2)
    ),
    'CNN': lambda: nn.Sequential(
        nn.Conv2d(1, 4, 3, padding=1), nn.ReLU(),
        nn.Flatten(), nn.Linear(4 * 8 * 8, 2)
    ),
}
loss_fn = nn.CrossEntropyLoss()

def train_one_epoch(model, loader, loss_fn, optimizer):
    model.train()
    total_loss = 0.0
    correct = 0
    total = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = loss_fn(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += x.size(0)
    return total_loss / total, correct / total

def evaluate(model, loader, loss_fn):
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits = model(x)
            loss = loss_fn(logits, y)
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += x.size(0)
    return total_loss / total, correct / total

results = {}
for name, build_model in model_builders.items():
    set_seed(SEED)
    model = build_model().to(device)
    train_loader = make_train_loader(SEED)
    # TODO: optimizer를 만들고 3 epoch 학습한 뒤 evaluate 결과를 저장하세요.
    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(3):
        train_loss, train_acc = train_one_epoch(model, train_loader, loss_fn, optimizer)
        valid_loss, valid_acc = evaluate(model, valid_loader, loss_fn)
    results[name] = {'valid_loss': {valid_loss}, 'valid_acc': {valid_acc}}
print(results)


{'MLP': {'valid_loss': {0.532492995262146}, 'valid_acc': {1.0}}, 'CNN': {'valid_loss': {0.5070704817771912}, 'valid_acc': {1.0}}}


### 해설 및 실행 결과 해석

- 같은 데이터·epoch 수뿐 아니라 모델 초기화 seed와 batch 순서도 각 모델 앞에서 같은 값으로 다시 설정했습니다. toy data에서는 두 모델 모두 단순한 선 패턴을 빠르게 학습할 수 있지만, 실제 이미지에서는 CNN이 공간 구조를 활용한다는 차이가 중요합니다.


## 문제 3. 비교 결과에서 best 모델 고르기

결과 dictionary에서 validation loss가 가장 낮은 모델을 찾습니다.


In [8]:
results = {
    'MLP': {'valid_loss': 0.42, 'valid_acc': 0.875},
    'CNN': {'valid_loss': 0.31, 'valid_acc': 1.0},
}
# TODO: valid_loss 기준 best 모델명을 찾으세요.
best_name = min(results, key=lambda x: results[x]['valid_loss'])
print('best:', best_name, results[best_name])


best: CNN {'valid_loss': 0.31, 'valid_acc': 1.0}


### 해설 및 실행 결과 해석

- 미리 정한 validation loss 기준으로는 CNN이 best입니다. 실제 리포트에서는 같은 선택 epoch의 accuracy와 모델 크기, 학습 시간, 데이터 특성도 함께 기록합니다.
